In [2]:
from pathlib import Path
from datetime import datetime, timezone
import shutil, json
from google.colab import files
import pandas as pd

In [3]:
!nvidia-smi

Wed Mar 11 17:57:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
%cd /content
!rm -rf llm-ml-educational-assistant
!git clone https://github.com/Nusultan11/llm-ml-educational-assistant.git
%cd /content/llm-ml-educational-assistant
!git log -1 --oneline

/content
Cloning into 'llm-ml-educational-assistant'...
remote: Enumerating objects: 289, done.
remote: Counting objects: 100% (289/289), done.
remote: Compressing objects: 100% (205/205), done.
remote: Total 289 (delta 123), reused 235 (delta 69), pack-reused 0 (from 0)
Receiving objects: 100% (289/289), 273.76 KiB | 2.24 MiB/s, done.
Resolving deltas: 100% (123/123), done.
/content/llm-ml-educational-assistant
91cbd8d (HEAD -> main, origin/main, origin/HEAD) Add retrieval ablation runner and utils


In [5]:
!pip install -q -r requirements.txt
!pip install -q -e .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 81.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llm-ml-assistant (pyproject.toml) ... done


In [6]:
root = Path("/content/llm-ml-educational-assistant")
print("run_retrieval_ablation.py exists:", (root / "scripts" / "run_retrieval_ablation.py").exists())

run_retrieval_ablation.py exists: True


In [7]:
!python scripts/prepare_datasets.py --out-dir data/processed --max-openassistant 30000 --max-dolly 15000
!python scripts/clean_processed_datasets.py --in-dir data/processed --out-dir data/processed_v2_clean_plus --rag-docs-dir data/rag_docs_v2_clean_plus --min-rag-chars 120 --min-instruction-chars 10 --min-response-chars 80 --max-non-ascii-ratio 0.3

Loading OpenAssistant...
README.md: 10.2kB [00:00, 19.9MB/s]
data/train-00000-of-00001-b42a775f407cee(…): 100% 39.5M/39.5M [00:03<00:00, 13.1MB/s]
data/validation-00000-of-00001-134b8fd0c(…): 100% 2.08M/2.08M [00:01<00:00, 2.08MB/s]
Generating train split: 100% 84437/84437 [00:00<00:00, 156131.83 examples/s]
Generating validation split: 100% 4401/4401 [00:00<00:00, 153180.19 examples/s]
Loading Dolly...
README.md: 8.20kB [00:00, 15.9MB/s]
databricks-dolly-15k.jsonl: 100% 13.1M/13.1M [00:00<00:00, 16.3MB/s]
Generating train split: 100% 15011/15011 [00:00<00:00, 182661.55 examples/s]
Loading local StackOverflow JSONL (if exists)...
Loading local ArXiv JSONL (if exists)...
Done
{
  "rag_rows": 9235,
  "sft_rows": 5095,
  "rag_path": "data/processed/rag_corpus.jsonl",
  "sft_path": "data/processed/sft_instructions.jsonl"
}
{
  "input_dir": "data/processed",
  "output_dir": "data/processed_v2_clean_plus",
  "rag_docs_dir": "data/rag_docs_v2_clean_plus",
  "params": {
    "min_rag_chars": 12

In [8]:
!python scripts/build_eval_from_cleaned.py \
  --sft-path data/processed_v2_clean_plus/sft_instructions.jsonl \
  --out data/processed_v2_clean_plus/eval_auto_qa.json \
  --max-items 200

{
  "sft_path": "data/processed_v2_clean_plus/sft_instructions.jsonl",
  "out_path": "data/processed_v2_clean_plus/eval_auto_qa.json",
  "items": 200,
  "seed": 42
}


In [9]:
!python scripts/run_retrieval_ablation.py \
  --base-config configs/colab_light.yaml \
  --rag-docs-dir data/rag_docs_v2_clean_plus \
  --processed-clean-dir data/processed_v2_clean_plus \
  --eval data/processed_v2_clean_plus/eval_auto_qa.json \
  --chunk-sizes 520 \
  --chunk-overlaps 80 \
  --top-ks 3,5,8 \
  --retrieval-modes hybrid \
  --tag-prefix colab_v2_topk

Run id: 20260311T175853Z
Profiles: 1 | Variants: 3

=== Build profile 1/1 [('hybrid', 520, 80)] ===
/usr/bin/python3 /content/llm-ml-educational-assistant/scripts/run_local_pipeline.py --config reports/retrieval_metrics/ablation/20260311T175853Z/configs/01_colab_v2_topk_hybrid_c520_o80_k3.yaml --artifacts-dir artifacts --processed-clean-dir data/processed_v2_clean_plus --rag-docs-dir data/rag_docs_v2_clean_plus --skip-prepare --snapshot-label 20260311T175853Z_01_colab_v2_topk_hybrid_c520_o80_k3 --snapshot-notes 'automated retrieval ablation run; profile=('"'"'hybrid'"'"', 520, 80); seed_tag=colab_v2_topk_hybrid_c520_o80_k3' --skip-clean

=== Build index ===
/usr/bin/python3 -m llm_ml_assistant.cli index --config reports/retrieval_metrics/ablation/20260311T175853Z/configs/01_colab_v2_topk_hybrid_c520_o80_k3.yaml --data-dir data/rag_docs_v2_clean_plus --artifacts-dir artifacts --rebuild
modules.json: 100% 349/349 [00:00<00:00, 1.87MB/s]
config_sentence_transformers.json: 100% 124/124 [00

In [10]:
!python scripts/run_retrieval_ablation.py \
  --base-config configs/colab_light.yaml \
  --rag-docs-dir data/rag_docs_v2_clean_plus \
  --processed-clean-dir data/processed_v2_clean_plus \
  --eval data/processed_v2_clean_plus/eval_auto_qa.json \
  --chunk-sizes 420,520,700 \
  --chunk-overlaps 80 \
  --top-ks 5 \
  --retrieval-modes hybrid \
  --tag-prefix colab_v2_chunk

Run id: 20260311T180351Z
Profiles: 3 | Variants: 3

=== Build profile 1/3 [('hybrid', 420, 80)] ===
/usr/bin/python3 /content/llm-ml-educational-assistant/scripts/run_local_pipeline.py --config reports/retrieval_metrics/ablation/20260311T180351Z/configs/01_colab_v2_chunk_hybrid_c420_o80_k5.yaml --artifacts-dir artifacts --processed-clean-dir data/processed_v2_clean_plus --rag-docs-dir data/rag_docs_v2_clean_plus --skip-prepare --snapshot-label 20260311T180351Z_01_colab_v2_chunk_hybrid_c420_o80_k5 --snapshot-notes 'automated retrieval ablation run; profile=('"'"'hybrid'"'"', 420, 80); seed_tag=colab_v2_chunk_hybrid_c420_o80_k5' --skip-clean

=== Build index ===
/usr/bin/python3 -m llm_ml_assistant.cli index --config reports/retrieval_metrics/ablation/20260311T180351Z/configs/01_colab_v2_chunk_hybrid_c420_o80_k5.yaml --data-dir data/rag_docs_v2_clean_plus --artifacts-dir artifacts --rebuild
Loading weights: 100% 199/199 [00:00<00:00, 929.99it/s, Materializing param=pooler.dense.weight]
B

In [11]:
!python scripts/run_retrieval_ablation.py \
  --base-config configs/colab_light.yaml \
  --rag-docs-dir data/rag_docs_v2_clean_plus \
  --processed-clean-dir data/processed_v2_clean_plus \
  --eval data/processed_v2_clean_plus/eval_auto_qa.json \
  --chunk-sizes 700 \
  --chunk-overlaps 40,80,120,160 \
  --top-ks 5 \
  --retrieval-modes hybrid \
  --tag-prefix colab_v2_overlap

Run id: 20260311T181627Z
Profiles: 4 | Variants: 4

=== Build profile 1/4 [('hybrid', 700, 40)] ===
/usr/bin/python3 /content/llm-ml-educational-assistant/scripts/run_local_pipeline.py --config reports/retrieval_metrics/ablation/20260311T181627Z/configs/01_colab_v2_overlap_hybrid_c700_o40_k5.yaml --artifacts-dir artifacts --processed-clean-dir data/processed_v2_clean_plus --rag-docs-dir data/rag_docs_v2_clean_plus --skip-prepare --snapshot-label 20260311T181627Z_01_colab_v2_overlap_hybrid_c700_o40_k5 --snapshot-notes 'automated retrieval ablation run; profile=('"'"'hybrid'"'"', 700, 40); seed_tag=colab_v2_overlap_hybrid_c700_o40_k5' --skip-clean

=== Build index ===
/usr/bin/python3 -m llm_ml_assistant.cli index --config reports/retrieval_metrics/ablation/20260311T181627Z/configs/01_colab_v2_overlap_hybrid_c700_o40_k5.yaml --data-dir data/rag_docs_v2_clean_plus --artifacts-dir artifacts --rebuild
Loading weights: 100% 199/199 [00:00<00:00, 1065.05it/s, Materializing param=pooler.dense.

In [12]:
root = Path("/content/llm-ml-educational-assistant")
bundle = root / "colab_export_final"
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir(parents=True, exist_ok=True)

# reports
shutil.copytree(root / "reports" / "retrieval_metrics", bundle / "retrieval_metrics", dirs_exist_ok=True)

# cleaning summary
clean = root / "data" / "processed_v2_clean_plus" / "cleaning_summary.json"
if clean.exists():
    shutil.copy2(clean, bundle / "cleaning_summary.json")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
zip_path = shutil.make_archive(f"/content/colab_final_logs_{stamp}", "zip", bundle)
print(zip_path)
files.download(zip_path)


/content/colab_final_logs_20260311T183312Z.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>